# llminfer Advanced Notebook (Colab)

Focus: advanced controls and performance tuning.
- HF loading options,
- compile + CUDA-graph toggles,
- speculative decoding controls,
- continuous batching,
- benchmark artifacts,
- optional vLLM + TP/PP comparison.



## 0) Runtime
In Colab, choose **Runtime -> Change runtime type -> GPU** before running.


In [ ]:
!nvidia-smi

import os
import platform
import torch

print('Python :', platform.python_version())
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
else:
    print('Warning: CUDA is not enabled. Use a GPU runtime for realistic results.')


## 1) Clone + install

In [ ]:
REPO_URL = 'https://github.com/nickforce989/llminfer.git'
TARGET_DIR = '/content/llminfer'

import os
import shutil

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

!git clone -q {REPO_URL} {TARGET_DIR}
%cd {TARGET_DIR}

!pip -q install -U pip
!pip -q install -e ".[serve,peft]"


## 2) Helpers

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

def header(title: str):
    print('\n' + '=' * 20 + f' {title} ' + '=' * 20)

def show_image(path: str):
    p = Path(path)
    if p.exists():
        print(f'[plot] {p}')
        display(Image(filename=str(p)))
    else:
        print(f'[missing plot] {p}')


## 3) HF loading controls

In [ ]:
from llminfer import InferenceEngine, EngineConfig
from llminfer.config import Backend

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

cfg = EngineConfig(
    model_name=MODEL,
    backend=Backend.EAGER,
    hf_revision='main',
    hf_local_files_only=False,
    hf_trust_remote_code=True,
    hf_cache_dir='/content/.cache/huggingface',
)

engine = InferenceEngine(cfg)
engine.load()
header('Engine info')
print(engine.info())


## 4) Constrained + speculative decoding knobs

In [ ]:
res = engine.run(
    'Explain GPUs for deep learning in 4 short bullet points.',
    max_new_tokens=128,
    temperature=0.2,
    no_repeat_ngram_size=3,
    bad_words=['joking', 'not sure'],
    stop_sequences=['\n\n'],
    seed=42,
    speculative_num_assistant_tokens=8,
    speculative_confidence_threshold=0.4,
)
header('Advanced decode result')
print(res.generated_text)
print('\nStats:', res.stats)


## 5) Compiled backend + CUDA graph controls

In [ ]:
from llminfer.config import Backend

cfg_compiled = EngineConfig(
    model_name='facebook/opt-125m',
    backend=Backend.COMPILED,
    compile_mode='reduce-overhead',
    compile_dynamic=True,
    compile_fullgraph=False,
    compile_cudagraphs=True,
)

engine_comp = InferenceEngine(cfg_compiled)
engine_comp.load()
engine_comp.warmup(prompt='Warmup prompt', n=1)

res_comp = engine_comp.run('What are CUDA graphs?', max_new_tokens=64, temperature=0.2)
header('Compiled output')
print(res_comp.generated_text)
print('\nStats:', res_comp.stats)


## 6) Continuous batching micro-demo

In [ ]:
prompts = [
    'One sentence on NVLink.',
    'One sentence on HBM.',
    'One sentence on tensor cores.',
    'One sentence on FlashAttention.',
]

out = engine.run_batch_continuous(prompts, max_new_tokens=40, temperature=0.2)
header('Continuous batching responses')
for i, r in enumerate(out, 1):
    print(f'{i}. {r.generated_text.strip()}')


## 7) Benchmark + artifact suite

In [ ]:
from llminfer import Benchmarker

bm = Benchmarker(engine_comp)

try:
    bench = bm.run(
        batch_sizes=[1, 2, 4],
        num_runs=3,
        warmup_runs=1,
        max_new_tokens=48,
        use_continuous_batching=True,
    )
except AssertionError as exc:
    print('[warn] Compiled backend + continuous batching can fail on some Colab torch builds.')
    print('[warn] Falling back to non-continuous benchmark for compiled backend.')
    print('[warn] Error:', exc)
    bench = bm.run(
        batch_sizes=[1, 2, 4],
        num_runs=3,
        warmup_runs=1,
        max_new_tokens=48,
        use_continuous_batching=False,
    )

bench.print_summary()

outdir = Path('benchmarks/advanced')
outdir.mkdir(parents=True, exist_ok=True)
bench.save_json(str(outdir / 'benchmark.json'))
bench.save_csv(str(outdir / 'benchmark.csv'))
plots = bench.plot_suite(str(outdir), prefix='advanced')
print('Saved plots:', plots)

for name in ['advanced_dashboard.png', 'advanced_throughput.png', 'advanced_latency.png', 'advanced_memory.png']:
    show_image(str(outdir / name))


## 8) Optional vLLM + tensor parallel benchmark

In [ ]:
RUN_VLLM = False  # set True to install and run vLLM compare

if RUN_VLLM:
    import os
    import platform
    import subprocess
    import sys

    print('Python:', platform.python_version())
    try:
        import torch
        print('Torch before install:', torch.__version__, 'CUDA:', torch.version.cuda)
    except Exception:
        print('Torch not importable before install')

    # Avoid editable extras here; force-refresh vLLM wheel in Colab runtime.
    !pip -q uninstall -y vllm
    !pip -q install --upgrade --force-reinstall --no-cache-dir vllm

    # Fix common Colab dependency side-effects from pip resolver.
    !pip -q install jedi "protobuf<6"

    try:
        import vllm  # noqa: F401
        import torch
        print('vLLM import: OK')
        print('Torch after install:', torch.__version__, 'CUDA:', torch.version.cuda)
    except Exception as exc:
        print('[error] vLLM import failed:', exc)
        print('[hint] This is usually a torch/vLLM ABI mismatch in the current runtime.')
        print('[hint] Try restarting runtime and rerunning the notebook from the top.')
        raise

    from llminfer.benchmark import BackendComparison
    from llminfer.config import Backend, QuantMode

    cmp = BackendComparison(
        model_name='facebook/opt-125m',
        backends=[Backend.EAGER, Backend.VLLM],
        quant_mode=QuantMode.NONE,
        tensor_parallel_size=1,
        pipeline_parallel_size=1,
        paged_kv=True,
        page_size_tokens=16,
    )
    cmp_results = cmp.run(batch_sizes=[1, 2, 4], num_runs=2, max_new_tokens=40, use_continuous_batching=True)
    cmp.print_table(cmp_results)

    cmp_dir = Path('benchmarks/advanced_vllm')
    cmp_dir.mkdir(parents=True, exist_ok=True)
    cmp.plot(cmp_results, str(cmp_dir / 'comparison.png'))
    show_image(str(cmp_dir / 'comparison.png'))
else:
    print('Skipping vLLM section. Set RUN_VLLM=True to run it.')


## 9) Cleanup

In [ ]:
for obj_name in ['engine', 'engine_comp']:
    if obj_name in globals():
        try:
            globals()[obj_name].unload()
        except Exception:
            pass
print('Done.')
